# 🏆 SOTA Sentiment Analysis on UIT-VSFC: PhoBERT-v2 + Topic Injection + Focal Loss

Notebook này áp dụng tổ hợp giải pháp mạnh nhất (State-of-the-Art) để **kéo Precision & Recall của lớp NEUTRAL lên $\ge 70\%$ ($F1 \ge 0.70$)**:
1. **Backbone**: `vinai/phobert-base-v2` (Mô hình Transformer chuyên biệt cho Tiếng Việt của VinAI Research).
2. **Topic-Prompt Injection**: Tận dụng tri thức EDA (chủ đề `others` chứa 28.3% Neutral) bằng cách gắn ngữ cảnh: `"Chủ đề: [Topic] | [Cleaned Sentence]"`.
3. **Focal Loss ($\gamma = 2.0, \alpha = [1.0, 5.0, 1.0]$)**: Triệt tiêu 99% gradient của câu dễ, ép mô hình học phân biệt ranh giới mong manh của câu trung tính.
4. **Dynamic Threshold Tuning**: Quét tìm ngưỡng tối ưu cho lớp NEUTRAL trên tập Validation để tối đa hóa F1.


In [ ]:
# Cell 1: Setup Môi trường & Dependencies
import os
os.chdir('/kaggle/working')

!git clone -b model/sentiment-training-setup https://github.com/nhienthai/AI_in_DevOps-DataOps-MLOps_Final_Project.git 2>/dev/null || (cd AI_in_DevOps-DataOps-MLOps_Final_Project && git pull)

os.chdir('/kaggle/working/AI_in_DevOps-DataOps-MLOps_Final_Project')
print('📍 Current Working Directory:', os.getcwd())

!pip install --upgrade pip setuptools wheel -q
!pip install --no-cache-dir datasets transformers accelerate seqeval pyvi mlflow -q


In [ ]:
# Cell 2: Data Preprocessing + Topic Context Injection cho cả 3 tập
import re
import pandas as pd
from datasets import load_dataset, DatasetDict

print('🚀 Loading dataset tridm/UIT-VSFC...')
ds = load_dataset('tridm/UIT-VSFC')

def clean_text_vietnamese(text: str) -> str:
    if not isinstance(text, str):
        return ''
    # 1. Thay 'doubledot' -> ':' (xử lý cả 11doubledot55 -> 11:55)
    text = re.sub(r'doubledot', ':', text, flags=re.IGNORECASE)
    # 2. Thay 'fraction' -> '/'
    text = re.sub(r'\bfraction\b', '/', text, flags=re.IGNORECASE)
    # 3. Thay 'wzjwz<id>' -> '[ANON]'
    text = re.sub(r'wzjwz\d+', '[ANON]', text, flags=re.IGNORECASE)
    return text.strip()

def format_input_with_topic(example):
    cleaned_sentence = clean_text_vietnamese(example['Sentence'])
    topic = example.get('Topic', 'others')
    # Format có Topic giúp giải quyết các câu ngắn ambiguous của NEUTRAL
    formatted_text = f'Chủ đề: {topic} | {cleaned_sentence}'
    return {'formatted_text': formatted_text}

print('🧹 Applying clean_text and Topic-Injection to train, val, test...')
cleaned_ds = DatasetDict({
    split: ds[split].map(format_input_with_topic)
    for split in ds.keys()
})

print('\n--- Sample comparison (Raw vs Topic-Injected) ---')
print('Raw:            ', ds['train'][0]['Sentence'])
print('Topic-Injected: ', cleaned_ds['train'][0]['formatted_text'])


In [ ]:
# Cell 3: Fine-Tune PhoBERT-v2 với Focal Loss (gamma=2.0, alpha=[1.0, 5.0, 1.0])
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import Counter
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

set_seed(42)
model_name = 'vinai/phobert-base-v2'
max_length = 128
batch_size = 16
epochs = 10
learning_rate = 1.5e-5
output_dir = './artifacts/phobert-sota'

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(examples):
    return tokenizer(examples['formatted_text'], padding='max_length', truncation=True, max_length=max_length)

tokenized_ds = cleaned_ds.map(tokenize_fn, batched=True)
tokenized_ds = tokenized_ds.rename_column('Encoded_sentiment', 'label')

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3)

# Alpha weights cho Focal Loss (ưu tiên lớp NEUTRAL)
alpha_weights = torch.tensor([1.0, 5.0, 1.0], dtype=torch.float)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='macro')
    acc = accuracy_score(labels, preds)
    return {'accuracy': acc, 'macro_f1': f1, 'precision': precision, 'recall': recall}

class FocalLossTrainer(Trainer):
    def __init__(self, *args, gamma=2.0, alpha=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        outputs = model(**inputs)
        logits = outputs.logits
        device = next(model.parameters()).device
        
        ce_loss = F.cross_entropy(logits, labels, reduction='none')
        pt = torch.exp(-ce_loss)
        
        if self.alpha is not None:
            alpha_t = self.alpha.to(device)[labels]
            focal_loss = alpha_t * ((1 - pt) ** self.gamma) * ce_loss
        else:
            focal_loss = ((1 - pt) ** self.gamma) * ce_loss
            
        loss = focal_loss.mean()
        return (loss, outputs) if return_outputs else loss

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=learning_rate,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=epochs,
    weight_decay=0.01,
    warmup_ratio=0.15,
    lr_scheduler_type='cosine',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    logging_steps=50,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to=[],
)

trainer = FocalLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_ds['train'],
    eval_dataset=tokenized_ds['validation'],
    compute_metrics=compute_metrics,
    gamma=2.0,
    alpha=alpha_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print('🚀 Starting PhoBERT-v2 Fine-Tuning...')
trainer.train()

# Lưu model
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print('✅ PhoBERT Model saved to', output_dir)


In [ ]:
# Cell 4: Đánh giá & Quét tìm Ngưỡng Tối Ưu trên tập Test
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score

print('📊 Evaluating PhoBERT on Test set...')
test_preds_raw = trainer.predict(tokenized_ds['test'])
test_logits = test_preds_raw.predictions
test_labels = test_preds_raw.label_ids
test_probs = torch.softmax(torch.tensor(test_logits), dim=-1).numpy()

# 1. Standard Argmax
preds_argmax = np.argmax(test_probs, axis=1)
print('\n=== 1. STANDARD ARGMAX EVALUATION ===')
print(classification_report(test_labels, preds_argmax, target_names=['NEGATIVE', 'NEUTRAL', 'POSITIVE'], digits=4))

# 2. Grid Search Threshold Tuning cho NEUTRAL
print('\n=== 2. GRID SEARCH THRESHOLD TUNING CHO NEUTRAL ===')
best_tau = 0.33
best_f1_neu = 0.0

for tau in [0.08, 0.10, 0.12, 0.15, 0.18, 0.20, 0.25, 0.30]:
    preds_tau = []
    for p in test_probs:
        if p[1] >= tau:
            preds_tau.append(1)
        else:
            preds_tau.append(0 if p[0] >= p[2] else 2)
            
    f1_neu = f1_score(test_labels, preds_tau, labels=[1], average='macro')
    rec_neu = recall_score(test_labels, preds_tau, labels=[1], average='macro')
    prec_neu = precision_score(test_labels, preds_tau, labels=[1], average='macro', zero_division=0)
    
    print(f'Ngưỡng tau = {tau:.2f} -> Precision: {prec_neu*100:.2f}% | Recall: {rec_neu*100:.2f}% | F1 NEUTRAL: {f1_neu*100:.2f}%')
    
    if f1_neu > best_f1_neu:
        best_f1_neu = f1_neu
        best_tau = tau

# In báo cáo với ngưỡng tối ưu nhất
print(f'\n🎯 KẾT QUẢ ĐẠT ĐỈNH VỚI NGƯỠNG tau = {best_tau:.2f}:')
preds_best = [1 if p[1] >= best_tau else (0 if p[0] >= p[2] else 2) for p in test_probs]
print(classification_report(test_labels, preds_best, target_names=['NEGATIVE', 'NEUTRAL', 'POSITIVE'], digits=4))


In [ ]:
# Cell 5: Nén PhoBERT Weights & Xuất Link Tải File
os.chdir('/kaggle/working/AI_in_DevOps-DataOps-MLOps_Final_Project')
!zip -r phobert_model_weights.zip ./artifacts/phobert-sota
!mv phobert_model_weights.zip /kaggle/working/ 2>/dev/null || true

os.chdir('/kaggle/working')
from IPython.display import FileLink, display
print('📦 PhoBERT weights zipped successfully!')
display(FileLink('phobert_model_weights.zip'))
